# 🚲 Cyclistic Bike-Share
## Análisis del comportamiento de miembros y usuarios ocasionales

Este notebook documenta el proceso de **preparación, limpieza y transformación de datos** utilizado para el caso práctico de Cyclistic.

El análisis utiliza **12 meses de datos públicos de Divvy Bike-Share de Chicago**, correspondientes al período comprendido entre **agosto de 2025 y julio de 2026**.

### 🎯 Objetivo

Preparar un dataset limpio y confiable que permita analizar las diferencias de comportamiento entre:

- **`member`**: usuarios con membresía anual.
- **`casual`**: usuarios ocasionales.

El dataset resultante será utilizado posteriormente para realizar el **análisis exploratorio de datos (EDA)** y desarrollar visualizaciones en **Tableau**.

### 🛠️ Herramientas utilizadas

- Python
- Pandas
- Google Colab
- Tableau Public

---

## 🧹 Etapa 1 — Carga, limpieza y preparación de los datos

### 1. Configuración del período de análisis

En este paso se define la carpeta donde se encuentran los archivos CSV y el período exacto que se desea estudiar. Establecer las fechas desde el comienzo permite asegurar que el análisis se limite a **12 meses consecutivos** y evita incluir registros fuera del rango seleccionado.

### 2. Localizar los archivos CSV

El código busca automáticamente los archivos CSV disponibles en la carpeta definida y muestra sus nombres. Esta comprobación permite verificar que se están utilizando únicamente los archivos necesarios antes de comenzar el procesamiento.

In [ ]:
import pandas as pd
import glob
import os

# =========================================================
# CYCLISTIC 1.0
# ETAPA 1 — CARGA, LIMPIEZA Y PREPARACIÓN DEL DATASET
# =========================================================


# ---------------------------------------------------------
# 1. CONFIGURACIÓN
# ---------------------------------------------------------

DATA_DIR = "/content/"
OUTPUT_DIR = "/content/cyclistic_outputs"

# Período exacto que vamos a estudiar:
# 1 agosto 2025 -> 31 julio 2026

START_DATE = "2025-08-01"
END_DATE = "2026-08-01"   # límite exclusivo


# ---------------------------------------------------------
# 2. LOCALIZAR Y VALIDAR LOS CSV
# ---------------------------------------------------------

all_files = sorted(
    glob.glob(
        os.path.join(
            DATA_DIR,
            "*-divvy-tripdata.csv"
        )
    )
)

# Validar que estén exactamente los 12 meses esperados
expected_months = (
    pd.period_range(
        "2025-08",
        "2026-07",
        freq="M"
    )
    .strftime("%Y%m")
    .tolist()
)

expected_files = [
    f"{month}-divvy-tripdata.csv"
    for month in expected_months
]

found_files = [
    os.path.basename(file)
    for file in all_files
]

if found_files != expected_files:
    raise ValueError(
        "Los archivos encontrados no coinciden con los "
        "12 meses esperados.\n"
        f"Esperados: {expected_files}\n"
        f"Encontrados: {found_files}"
    )

print("Archivos CSV validados:\n")

for file in all_files:
    print(os.path.basename(file))

print(
    f"\nTotal de archivos validados: "
    f"{len(all_files)}"
)


### 3. Cargar y concatenar los archivos

Cada archivo mensual se carga en un DataFrame y posteriormente todos se combinan en un único conjunto de datos. También se registra el archivo de origen de cada fila mediante source_file, lo que facilita rastrear posibles errores o anomalías.

In [ ]:
# ---------------------------------------------------------
# 3. CARGAR Y CONCATENAR LOS ARCHIVOS
# ---------------------------------------------------------

df_list = []

for file in all_files:

    temp_df = pd.read_csv(file)

    # Registrar de qué archivo provino cada fila
    temp_df["source_file"] = os.path.basename(file)

    df_list.append(temp_df)


df = pd.concat(
    df_list,
    ignore_index=True
)

print(f"\nRegistros cargados: {len(df):,}")


Registros cargados: 6,037,968


### 4. Validación inicial de los datos

Antes de realizar modificaciones, se revisan las dimensiones del dataset, los nombres de las columnas, los tipos de datos y los valores faltantes. Esta exploración permite conocer la estructura de la información y detectar posibles problemas antes de comenzar la limpieza.

In [ ]:
# ---------------------------------------------------------
# 4. VALIDACIÓN INICIAL
# ---------------------------------------------------------

print("\nDimensiones:")
print(df.shape)

print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())


Dimensiones:
(6037968, 14)

Columnas:
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'source_file']

Tipos de datos:
ride_id                object
rideable_type          object
started_at             object
ended_at               object
start_station_name     object
start_station_id       object
end_station_name       object
end_station_id         object
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual          object
source_file            object
dtype: object

Valores nulos:
ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1273200
start_station_id      1273200
end_station_name      1336777
end_station_id        1336777
start_lat                   0
start_lng

### 5. Convertir las fechas

Las columnas `started_at` y `ended_at` se convierten al formato `datetime` de Pandas. Esto permite realizar operaciones con fechas y horas, como calcular la duración de cada viaje o extraer el día, mes y hora de inicio.

La opción `errors="coerce"` transforma fechas inválidas en valores nulos (`NaT`) para poder identificarlas y tratarlas posteriormente sin detener la ejecución del programa.

In [ ]:
# ---------------------------------------------------------
# 5. CONVERSIÓN DE FECHAS
# ---------------------------------------------------------

df["started_at"] = pd.to_datetime(
    df["started_at"],
    errors="coerce"
)

df["ended_at"] = pd.to_datetime(
    df["ended_at"],
    errors="coerce"
)


invalid_dates = (
    df["started_at"].isna() |
    df["ended_at"].isna()
).sum()

print(
    f"\nRegistros con fechas inválidas: "
    f"{invalid_dates:,}"
)


Registros con fechas inválidas: 0


###
**Eliminar fechas inválidas:**

Después de convertir las columnas de fecha y hora, se eliminan los registros que no poseen una fecha de inicio o finalización válida. Estos datos son necesarios para calcular correctamente la duración de cada viaje, por lo que los registros incompletos no pueden utilizarse en el análisis. La función .copy() crea una copia independiente del DataFrame limpio para continuar trabajando de forma segura.

In [ ]:
df = df.dropna(
    subset=["started_at", "ended_at"]
).copy()

### 6. Filtrar el período de análisis

Se conservan únicamente los viajes iniciados entre agosto de 2025 y julio de 2026. Este filtro funciona como una segunda capa de seguridad para garantizar que ningún registro perteneciente a otro período sea incluido accidentalmente.

In [ ]:
# ---------------------------------------------------------
# 6. DEFINIR EL PERÍODO DEL ANÁLISIS
# ---------------------------------------------------------

before_period_filter = len(df)

df = df[
    (df["started_at"] >= START_DATE) &
    (df["started_at"] < END_DATE)
].copy()

removed_outside_period = (
    before_period_filter - len(df)
)

print(
    f"\nRegistros fuera del período eliminados: "
    f"{removed_outside_period:,}"
)

print(
    f"Período final: "
    f"{df['started_at'].min()} -> "
    f"{df['started_at'].max()}"
)


Registros fuera del período eliminados: 128
Período final: 2025-08-01 00:00:32.506000 -> 2026-07-31 23:58:05.898000


### 7. Eliminar registros duplicados

Se utiliza `ride_id` como identificador único de cada viaje para detectar y eliminar registros repetidos.

Esta deduplicación se realiza después de concatenar todos los archivos, ya que un mismo viaje podría aparecer en dos archivos mensuales diferentes aunque cada archivo por separado no contenga duplicados.

In [ ]:
# ---------------------------------------------------------
# 7. ELIMINAR DUPLICADOS
# ---------------------------------------------------------

duplicates = df.duplicated(
    subset="ride_id",
    keep=False
)

print(
    f"\nFilas asociadas a ride_id duplicados: "
    f"{duplicates.sum():,}"
)


before_duplicates = len(df)

df = df.drop_duplicates(
    subset="ride_id",
    keep="first"
).copy()


removed_duplicates = (
    before_duplicates - len(df)
)

print(
    f"Duplicados eliminados: "
    f"{removed_duplicates:,}"
)


Filas asociadas a ride_id duplicados: 70
Duplicados eliminados: 35


### 8. Calcular la duración del viaje

La duración de cada viaje se obtiene restando la fecha y hora de inicio (started_at) de la fecha y hora de finalización (ended_at).

El resultado se convierte a minutos y se almacena en la variable: `ride_length_min`

Esta variable será fundamental para comparar cuánto tiempo utilizan las bicicletas los miembros y los usuarios ocasionales.

In [ ]:
# ---------------------------------------------------------
# 8. DURACIÓN DEL VIAJE
# ---------------------------------------------------------

df["ride_length_min"] = (
    df["ended_at"] - df["started_at"]
).dt.total_seconds() / 60

### 9. Filtrar duraciones inválidas

Se eliminan viajes cuya duración sea menor o igual a cero, ya que representan registros inconsistentes, y también aquellos superiores a 24 horas.

Este filtro busca reducir el efecto de registros atípicos o errores potenciales sin eliminar arbitrariamente viajes cortos que todavía podrían ser válidos.

In [ ]:
# ---------------------------------------------------------
# 9. FILTRAR DURACIONES INVÁLIDAS
# ---------------------------------------------------------

invalid_duration = (
    (df["ride_length_min"] <= 0) |
    (df["ride_length_min"] > 1440)
)

print(
    f"\nViajes con duración inválida: "
    f"{invalid_duration.sum():,}"
)


df = df[
    ~invalid_duration
].copy()


Viajes con duración inválida: 5,334


### 10. Eliminar estaciones de prueba o mantenimiento

Se identifican viajes relacionados con estaciones cuyo nombre contiene términos como TEST o HQ.

Estos registros pueden corresponder a pruebas técnicas, mantenimiento o movimientos internos y, por lo tanto, no representan necesariamente el comportamiento normal de los clientes.

Se revisan tanto las estaciones de inicio como las de finalización.

In [ ]:
# ---------------------------------------------------------
# 10. ESTACIONES DE PRUEBA / MANTENIMIENTO
# ---------------------------------------------------------

maintenance_pattern = r"TEST|HQ"

start_test = df["start_station_name"].str.contains(
    maintenance_pattern,
    case=False,
    na=False
)

end_test = df["end_station_name"].str.contains(
    maintenance_pattern,
    case=False,
    na=False
)

maintenance_rides = start_test | end_test

print(
    f"\nViajes asociados a estaciones TEST/HQ: "
    f"{maintenance_rides.sum():,}"
)


df = df[
    ~maintenance_rides
].copy()


Viajes asociados a estaciones TEST/HQ: 2


### 11. Crear nuevas variables para el análisis

A partir de `started_at` se crean variables adicionales que facilitan el análisis:

*   `year`: año del viaje.
*   `month`: nombre del mes.
*   `month_num`: número del mes.
*   `year_month`: combinación de año y mes.
*   `month_start`: fecha correspondiente al inicio del mes.
*   `day_of_week`: día de la semana.
*   `day_num`: número del día de la semana.
*   `start_hour`: hora de inicio.
*   `is_weekend`: identifica si el viaje ocurrió durante sábado o domingo.

Estas variables permiten estudiar patrones temporales sin modificar las fechas originales.

Especialmente importante es `year_month`, porque evita combinar accidentalmente, por ejemplo, **julio de 2025 con julio de 2026**.

In [ ]:
# ---------------------------------------------------------
# 11. INGENIERÍA DE VARIABLES
# ---------------------------------------------------------

df["year"] = df["started_at"].dt.year

df["month"] = df["started_at"].dt.month_name()

df["month_num"] = df["started_at"].dt.month

df["year_month"] = (
    df["started_at"]
    .dt.to_period("M")
    .astype(str)
)

df["month_start"] = (
    df["started_at"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

df["day_of_week"] = (
    df["started_at"]
    .dt.day_name()
)

df["day_num"] = (
    df["started_at"]
    .dt.dayofweek
)

df["start_hour"] = (
    df["started_at"]
    .dt.hour
)

df["is_weekend"] = (
    df["day_num"] >= 5
)

### 12. Auditoría final del dataset

Antes de guardar los datos, se realiza una última comprobación de calidad.

El código verifica:

*   cantidad final de registros;
*   número de viajes únicos;
*   duplicados restantes;
*   fecha mínima y máxima;
*   tipos de usuarios;
*   tipos de bicicletas;
*   cantidad de viajes por mes.

Esta etapa permite confirmar que el dataset cumple con las condiciones esperadas antes de utilizarlo en el análisis exploratorio.

In [ ]:
# ---------------------------------------------------------
# 12. VALIDACIÓN FINAL
# ---------------------------------------------------------

print("\n==============================")
print("RESUMEN DATASET LIMPIO")
print("==============================")

print(
    f"Registros finales: "
    f"{len(df):,}"
)

print(
    f"Ride IDs únicos: "
    f"{df['ride_id'].nunique():,}"
)

print(
    f"Duplicados restantes: "
    f"{df['ride_id'].duplicated().sum():,}"
)

print(
    f"Fecha mínima: "
    f"{df['started_at'].min()}"
)

print(
    f"Fecha máxima: "
    f"{df['started_at'].max()}"
)

print("\nTipos de usuario:")
print(
    df["member_casual"]
    .value_counts()
)

print("\nTipos de bicicleta:")
print(
    df["rideable_type"]
    .value_counts()
)

print("\nViajes por mes:")
print(
    df["year_month"]
    .value_counts()
    .sort_index()
)

# ---------------------------------------------------------
# VALIDACIONES AUTOMÁTICAS
# ---------------------------------------------------------

assert df["ride_id"].duplicated().sum() == 0
assert df["started_at"].min() >= pd.Timestamp(START_DATE)
assert df["started_at"].max() < pd.Timestamp(END_DATE)
assert df["year_month"].nunique() == 12
assert (df["ride_length_min"] > 0).all()
assert (df["ride_length_min"] <= 1440).all()
assert set(df["member_casual"].dropna().unique()) <= {
    "member",
    "casual"
}

print(
    "\nValidaciones automáticas superadas correctamente."
)


### 13. Exportar el dataset limpio

Finalmente, el DataFrame procesado se guarda como un nuevo archivo CSV.

De esta forma se conserva una versión limpia y reproducible del dataset que podrá utilizarse posteriormente para el **análisis exploratorio y la creación de visualizaciones en Tableau.**

In [ ]:
# ---------------------------------------------------------
# 13. EXPORTAR DATASET LIMPIO
# ---------------------------------------------------------

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

output_path = os.path.join(
    OUTPUT_DIR,
    "cyclistic_clean_v1.csv"
)

df.to_csv(
    output_path,
    index=False
)

print(
    f"\nDataset guardado en:\n"
    f"{output_path}"
)


También puede guardarse una versión comprimida .csv.gz para reducir considerablemente el espacio de almacenamiento sin perder información.

In [ ]:
gzip_path = os.path.join(
    OUTPUT_DIR,
    "cyclistic_clean_v1.csv.gz"
)

df.to_csv(
    gzip_path,
    index=False,
    compression="gzip"
)

print(
    f"Versión comprimida guardada en:\n"
    f"{gzip_path}"
)


# 📊 Etapa 2 — Análisis Exploratorio de Datos (EDA)

Una vez finalizada la limpieza y preparación de los datos, se inicia el análisis exploratorio con el objetivo de identificar diferencias en los patrones de uso entre los miembros anuales (`member`) y los usuarios ocasionales (`casual`).

El análisis se organiza en cuatro dimensiones principales:

- Volumen y frecuencia de viajes.
- Patrones temporales de uso.
- Duración de los viajes.
- Preferencias de bicicleta y estaciones.

## 1. Volumen de viajes por tipo de usuario

El primer análisis busca conocer cómo se distribuye el volumen total de viajes entre los miembros anuales (`member`) y los usuarios ocasionales (`casual`).

Para ello se calcula la cantidad total de viajes realizados por cada grupo y su participación porcentual sobre el total del dataset.

Este análisis permite dimensionar el peso de cada segmento dentro del servicio y sirve como punto de partida para las comparaciones posteriores.

In [ ]:
# ---------------------------------------------------------
# 1. VOLUMEN DE VIAJES POR TIPO DE USUARIO
# ---------------------------------------------------------

# Contar viajes por tipo de usuario
rides_by_user = (
    df["member_casual"]
    .value_counts()
    .rename_axis("user_type")
    .reset_index(name="total_rides")
)

# Calcular participación porcentual
rides_by_user["percentage"] = (
    rides_by_user["total_rides"]
    / rides_by_user["total_rides"].sum()
    * 100
)

# Redondear porcentaje
rides_by_user["percentage"] = (
    rides_by_user["percentage"]
    .round(2)
)

# Crear tabla de presentación

volume_summary = rides_by_user.copy()

volume_summary["total_rides"] = (
    volume_summary["total_rides"]
    .map("{:,}".format)
)

volume_summary["percentage"] = (
    volume_summary["percentage"]
    .map("{:.2f}%".format)
)

volume_summary

,user_type,total_rides,percentage
0,member,"3,885,831",64.42%
1,casual,"2,146,638",35.58%


**Cálculo del volumen de viajes**

El código utiliza `value_counts()` para contar cuántos viajes corresponden a cada categoría de la variable `member_casual`.

Posteriormente se calcula qué porcentaje representa cada grupo sobre el total de viajes analizados.

De esta forma podemos comparar no solamente la cantidad absoluta de viajes, sino también la participación relativa de miembros y usuarios ocasionales dentro del servicio.

### Hallazgo

Durante el período analizado se registraron **6.032.469 viajes**.

Los miembros anuales (`member`) realizaron **3.885.831 viajes**, equivalentes al **64,42 %** del total, mientras que los usuarios ocasionales (`casual`) realizaron **2.146.638 viajes**, representando el **35,58 %**.

Aunque los miembros generan la mayor parte de la actividad del servicio, los usuarios ocasionales representan más de un tercio de todos los viajes. Esto convierte al segmento `casual` en un grupo relevante para analizar posibles oportunidades de conversión hacia membresías anuales.

## 2. Evolución mensual y estacionalidad

En esta etapa se analiza cómo varía el número de viajes a lo largo de los 12 meses comprendidos entre **agosto de 2025 y julio de 2026**.

Además del volumen total, se compara la participación de miembros anuales (`member`) y usuarios ocasionales (`casual`) en cada mes.

Este análisis permite identificar posibles patrones estacionales y determinar si alguno de los dos grupos presenta cambios más pronunciados según la época del año.

In [ ]:
# ---------------------------------------------------------
# 2. EVOLUCIÓN MENSUAL DE LOS VIAJES
# ---------------------------------------------------------

monthly_rides = (
    df.groupby(
        ["month_start", "member_casual"]
    )
    .size()
    .reset_index(name="total_rides")
)

monthly_rides.head()

,month_start,member_casual,total_rides
0,2025-08-01,casual,337246
1,2025-08-01,member,452355
2,2025-09-01,casual,264772
3,2025-09-01,member,449202
4,2025-10-01,casual,223569


**Agrupación mensual de los viajes**

El código agrupa los registros según el mes de inicio del viaje (`month_start`) y el tipo de usuario (`member_casual`).

Posteriormente, `.size()` cuenta cuántos viajes fueron realizados por cada grupo durante cada mes.

Se utiliza `month_start` en lugar del nombre del mes para conservar una fecha real y garantizar que los meses permanezcan ordenados cronológicamente.

In [ ]:
# Crear tabla comparativa por mes

monthly_summary = (
    monthly_rides
    .pivot(
        index="month_start",
        columns="member_casual",
        values="total_rides"
    )
    .reset_index()
)

# Eliminar el nombre adicional creado por pivot
monthly_summary.columns.name = None

# Calcular total mensual
monthly_summary["total_rides"] = (
    monthly_summary["member"] +
    monthly_summary["casual"]
)

# Calcular participación de usuarios casuales
monthly_summary["casual_percentage"] = (
    monthly_summary["casual"]
    / monthly_summary["total_rides"]
    * 100
)

# Calcular participación de miembros
monthly_summary["member_percentage"] = (
    monthly_summary["member"]
    / monthly_summary["total_rides"]
    * 100
)

monthly_summary

,month_start,casual,member,total_rides,casual_percentage,member_percentage
0,2025-08-01,337246,452355,789601,42.710939,57.289061
1,2025-09-01,264772,449202,713974,37.084264,62.915736
2,2025-10-01,223569,421921,645490,34.635548,65.364452
3,2025-11-01,98871,257292,356163,27.760042,72.239958
4,2025-12-01,28005,112415,140420,19.943740,80.056260
5,2026-01-01,24656,112933,137589,17.920037,82.079963
6,2026-02-01,41088,160251,201339,20.407373,79.592627
7,2026-03-01,87631,229121,316752,27.665492,72.334508
8,2026-04-01,131223,316675,447898,29.297519,70.702481
9,2026-05-01,244654,408510,653164,37.456749,62.543251


**Comparación mensual entre tipos de usuario**

A partir de los datos agrupados se crea una tabla donde cada fila representa un mes y se muestran por separado los viajes realizados por miembros y usuarios ocasionales.

También se calcula el volumen total de viajes del mes y el porcentaje que representa cada grupo.

La participación porcentual es especialmente útil porque permite comparar el comportamiento de los usuarios independientemente de que el volumen general del sistema aumente o disminuya según la temporada.

In [ ]:
print(
    f"Cantidad de meses analizados: "
    f"{monthly_summary['month_start'].nunique()}"
)

print(
    f"Primer mes: "
    f"{monthly_summary['month_start'].min():%Y-%m}"
)

print(
    f"Último mes: "
    f"{monthly_summary['month_start'].max():%Y-%m}"
)

Cantidad de meses analizados: 12
Primer mes: 2025-08
Último mes: 2026-07


**Validación del período**

Antes de interpretar los resultados se verifica que el análisis contenga exactamente **12 meses consecutivos**, desde agosto de 2025 hasta julio de 2026.

Este control permite confirmar que no fueron incluidos accidentalmente meses pertenecientes a otro período.

In [ ]:
# Crear versión de presentación

monthly_display = monthly_summary.copy()

# Mostrar mes como YYYY-MM
monthly_display["month_start"] = (
    monthly_display["month_start"]
    .dt.strftime("%Y-%m")
)

# Formatear cantidades
for col in ["member", "casual", "total_rides"]:
    monthly_display[col] = (
        monthly_display[col]
        .map("{:,}".format)
    )

# Formatear porcentajes
monthly_display["member_percentage"] = (
    monthly_display["member_percentage"]
    .map("{:.2f}%".format)
)

monthly_display["casual_percentage"] = (
    monthly_display["casual_percentage"]
    .map("{:.2f}%".format)
)

monthly_display

,month_start,casual,member,total_rides,casual_percentage,member_percentage
0,2025-08,"337,246","452,355","789,601",42.71%,57.29%
1,2025-09,"264,772","449,202","713,974",37.08%,62.92%
2,2025-10,"223,569","421,921","645,490",34.64%,65.36%
3,2025-11,"98,871","257,292","356,163",27.76%,72.24%
4,2025-12,"28,005","112,415","140,420",19.94%,80.06%
5,2026-01,"24,656","112,933","137,589",17.92%,82.08%
6,2026-02,"41,088","160,251","201,339",20.41%,79.59%
7,2026-03,"87,631","229,121","316,752",27.67%,72.33%
8,2026-04,"131,223","316,675","447,898",29.30%,70.70%
9,2026-05,"244,654","408,510","653,164",37.46%,62.54%


### Hallazgo

El volumen de viajes presenta una marcada variación a lo largo del año.

El mes con mayor actividad fue **julio de 2026**, con **868.183 viajes**, mientras que **enero de 2026** registró el menor volumen, con **137.589 viajes**.

La participación de los usuarios ocasionales también muestra un comportamiento estacional más pronunciado. En **agosto de 2025**, los usuarios `casual` representaron el **42,71 %** de los viajes, mientras que en **enero de 2026** su participación descendió hasta el **17,92 %**.

Entre el valor mínimo y máximo existe una diferencia de **24,79 puntos porcentuales** en la participación de usuarios ocasionales.

Aunque ambos grupos reducen su actividad durante los meses de menor demanda, el segmento `casual` presenta una variación proporcional mucho mayor. Esto sugiere que su comportamiento está más condicionado por la estacionalidad que el de los miembros anuales.

Los meses con mayor participación de usuarios ocasionales coinciden con los períodos de mayor actividad general del sistema, lo que podría representar una oportunidad para intensificar las campañas de conversión durante esos meses.

## 3. Comportamiento por día de la semana

En esta etapa se analiza cómo se distribuyen los viajes entre los diferentes días de la semana para los miembros anuales (`member`) y los usuarios ocasionales (`casual`).

El objetivo es identificar si ambos grupos presentan patrones de uso similares o si concentran su actividad en días diferentes.

Además, se realizará una comparación entre **días laborales y fines de semana** para medir con mayor claridad estas diferencias.

### Continuidad del análisis\n\nEl análisis exploratorio continúa directamente sobre el DataFrame limpio `df` generado en la etapa de procesamiento. No es necesario volver a cargar el dataset desde Google Drive, lo que evita dependencias de rutas personales y mejora la reproducibilidad del notebook.

In [ ]:
# ---------------------------------------------------------
# 3. VIAJES POR DÍA DE LA SEMANA
# ---------------------------------------------------------

# Orden correcto de los días
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

# Contar viajes por día y tipo de usuario
rides_by_day = (
    df.groupby(
        ["day_of_week", "member_casual"]
    )
    .size()
    .reset_index(name="total_rides")
)

# Convertir day_of_week en categoría ordenada
rides_by_day["day_of_week"] = pd.Categorical(
    rides_by_day["day_of_week"],
    categories=day_order,
    ordered=True
)

# Ordenar resultados
rides_by_day = rides_by_day.sort_values(
    ["day_of_week", "member_casual"]
)

rides_by_day

,day_of_week,member_casual,total_rides
2,Monday,casual,243861
3,Monday,member,538584
10,Tuesday,casual,237777
11,Tuesday,member,614189
12,Wednesday,casual,245787
13,Wednesday,member,622773
8,Thursday,casual,267275
9,Thursday,member,619291
0,Friday,casual,339142
1,Friday,member,581398


**Distribución de viajes por día de la semana**

El código agrupa los viajes según el día de la semana y el tipo de usuario.

Se establece manualmente el orden de los días desde lunes hasta domingo para evitar que Pandas los ordene alfabéticamente.

El resultado permite comparar cuántos viajes realizan los miembros y los usuarios ocasionales en cada día de la semana.

In [ ]:
# Crear tabla comparativa por día

day_summary = (
    rides_by_day
    .pivot(
        index="day_of_week",
        columns="member_casual",
        values="total_rides"
    )
    .reset_index()
)

day_summary.columns.name = None

day_summary["total_rides"] = (
    day_summary["member"] +
    day_summary["casual"]
)

day_summary

,day_of_week,casual,member,total_rides
0,Monday,243861,538584,782445
1,Tuesday,237777,614189,851966
2,Wednesday,245787,622773,868560
3,Thursday,267275,619291,886566
4,Friday,339142,581398,920540
5,Saturday,456408,494433,950841
6,Sunday,356388,415163,771551


In [ ]:
# Crear versión de presentación

day_display = day_summary.copy()

for col in ["member", "casual", "total_rides"]:
    day_display[col] = (
        day_display[col]
        .map("{:,}".format)
    )

day_display

,day_of_week,casual,member,total_rides
0,Monday,"243,861","538,584","782,445"
1,Tuesday,"237,777","614,189","851,966"
2,Wednesday,"245,787","622,773","868,560"
3,Thursday,"267,275","619,291","886,566"
4,Friday,"339,142","581,398","920,540"
5,Saturday,"456,408","494,433","950,841"
6,Sunday,"356,388","415,163","771,551"


**Comparación entre miembros y usuarios ocasionales**

La tabla reorganiza los datos para mostrar en una misma fila la cantidad de viajes realizados por miembros y usuarios ocasionales en cada día de la semana.

Esto facilita la comparación directa entre ambos segmentos y permite detectar qué días concentran la mayor actividad.

In [ ]:
# ---------------------------------------------------------
# PARTICIPACIÓN DE CADA DÍA DENTRO DE CADA GRUPO
# ---------------------------------------------------------

day_percentage = rides_by_day.copy()

day_percentage["percentage"] = (
    day_percentage["total_rides"]
    / day_percentage.groupby(
        "member_casual"
    )["total_rides"].transform("sum")
    * 100
)

day_percentage["percentage"] = (
    day_percentage["percentage"]
    .round(2)
)

day_percentage

,day_of_week,member_casual,total_rides,percentage
2,Monday,casual,243861,11.36
3,Monday,member,538584,13.86
10,Tuesday,casual,237777,11.08
11,Tuesday,member,614189,15.81
12,Wednesday,casual,245787,11.45
13,Wednesday,member,622773,16.03
8,Thursday,casual,267275,12.45
9,Thursday,member,619291,15.94
0,Friday,casual,339142,15.80
1,Friday,member,581398,14.96


**Participación semanal dentro de cada tipo de usuario**

Debido a que los miembros realizan una mayor cantidad total de viajes, comparar únicamente valores absolutos podría ocultar diferencias en el comportamiento relativo de cada grupo.

Por este motivo, se calcula qué porcentaje de los viajes de cada segmento ocurre en cada día de la semana.

Esta medida permite comparar los patrones de uso de miembros y usuarios ocasionales independientemente del tamaño de cada grupo.

In [ ]:
# ---------------------------------------------------------
# 3.1 DÍAS LABORALES VS. FIN DE SEMANA
# ---------------------------------------------------------

weekpart_summary = (
    df.assign(
        week_part=df["is_weekend"].map({
            False: "Weekday",
            True: "Weekend"
        })
    )
    .groupby(
        ["member_casual", "week_part"]
    )
    .size()
    .reset_index(name="total_rides")
)

weekpart_summary

,member_casual,week_part,total_rides
0,casual,Weekday,1333842
1,casual,Weekend,812796
2,member,Weekday,2976235
3,member,Weekend,909596


**Comparación entre días laborales y fines de semana**

Los viajes se clasifican en dos grupos:

- `Weekday`: lunes a viernes.
- `Weekend`: sábado y domingo.

Esta comparación permite medir de forma más directa si alguno de los segmentos concentra una mayor proporción de su actividad durante el fin de semana.

In [ ]:
# Calcular porcentaje dentro de cada tipo de usuario

weekpart_summary["percentage"] = (
    weekpart_summary["total_rides"]
    / weekpart_summary.groupby(
        "member_casual"
    )["total_rides"].transform("sum")
    * 100
)

weekpart_summary["percentage"] = (
    weekpart_summary["percentage"]
    .round(2)
)

weekpart_summary

,member_casual,week_part,total_rides,percentage
0,casual,Weekday,1333842,62.14
1,casual,Weekend,812796,37.86
2,member,Weekday,2976235,76.59
3,member,Weekend,909596,23.41


In [ ]:
weekpart_display = weekpart_summary.copy()

weekpart_display["total_rides"] = (
    weekpart_display["total_rides"]
    .map("{:,}".format)
)

weekpart_display["percentage"] = (
    weekpart_display["percentage"]
    .map("{:.2f}%".format)
)

weekpart_display

,member_casual,week_part,total_rides,percentage
0,casual,Weekday,"1,333,842",62.14%
1,casual,Weekend,"812,796",37.86%
2,member,Weekday,"2,976,235",76.59%
3,member,Weekend,"909,596",23.41%


In [ ]:
# ---------------------------------------------------------
# PROMEDIO DIARIO: WEEKDAY VS WEEKEND
# ---------------------------------------------------------

weekpart_daily = weekpart_summary.copy()

weekpart_daily["number_of_days"] = (
    weekpart_daily["week_part"]
    .map({
        "Weekday": 5,
        "Weekend": 2
    })
)

weekpart_daily["avg_rides_per_day"] = (
    weekpart_daily["total_rides"]
    / weekpart_daily["number_of_days"]
)

weekpart_daily["avg_rides_per_day"] = (
    weekpart_daily["avg_rides_per_day"]
    .round(0)
    .astype(int)
)

weekpart_daily

,member_casual,week_part,total_rides,percentage,number_of_days,avg_rides_per_day
0,casual,Weekday,1333842,62.14,5,266768
1,casual,Weekend,812796,37.86,2,406398
2,member,Weekday,2976235,76.59,5,595247
3,member,Weekend,909596,23.41,2,454798


### Hallazgo 1 — Distribución por día de la semana

Los miembros anuales y los usuarios ocasionales presentan patrones diferentes según el día de la semana.

Los miembros (`member`) concentran su mayor actividad entre semana, alcanzando su máximo el **miércoles**, con **622.773 viajes**.

En cambio, los usuarios ocasionales (`casual`) registran su mayor volumen el **sábado**, con **456.408 viajes**.

Este contraste sugiere que los miembros mantienen un patrón de uso más estable durante los días laborales, mientras que los usuarios ocasionales muestran una mayor concentración de actividad durante el fin de semana.

### Hallazgo 2 — Días laborales vs. fines de semana

Al agrupar los viajes entre días laborales y fines de semana, la diferencia entre ambos grupos se vuelve todavía más clara.

Los miembros realizan el **76,59 %** de sus viajes de lunes a viernes y solamente el **23,41 %** durante sábado y domingo.

Los usuarios ocasionales presentan una distribución diferente: el **62,14 %** de sus viajes ocurre entre semana y el **37,86 %** durante el fin de semana.

Esto representa una diferencia de **14,45 puntos porcentuales** en la participación del fin de semana entre ambos segmentos.

Además, al considerar la diferente cantidad de días de cada período, el promedio diario de viajes de los usuarios ocasionales es mayor durante el fin de semana que durante los días laborales. Esto refuerza la importancia relativa de sábados y domingos dentro de su comportamiento de uso.

Estos resultados son compatibles con un uso más rutinario entre los miembros y un componente recreativo mayor entre los usuarios ocasionales. Sin embargo, el dataset no incluye el motivo declarado de los viajes, por lo que esta interpretación debe considerarse una inferencia.

## 4. Comportamiento por hora del día

En esta etapa se analiza la cantidad de viajes iniciados en cada hora del día para los miembros anuales (`member`) y los usuarios ocasionales (`casual`).

El objetivo es identificar posibles diferencias en los horarios de uso y determinar si ambos segmentos concentran su actividad en momentos diferentes del día.

In [ ]:
# ---------------------------------------------------------
# 4. VIAJES POR HORA DEL DÍA
# ---------------------------------------------------------

rides_by_hour = (
    df.groupby(
        ["start_hour", "member_casual"]
    )
    .size()
    .reset_index(name="total_rides")
)

rides_by_hour

,start_hour,member_casual,total_rides
0,0,casual,44454
1,0,member,36089
2,1,casual,29168
3,1,member,21978
4,2,casual,18628
5,2,member,13070
6,3,casual,10576
7,3,member,8457
8,4,casual,8484
9,4,member,9547


**Distribución de viajes por hora**

El código agrupa los viajes según la hora de inicio (`start_hour`) y el tipo de usuario.

Posteriormente se cuenta cuántos viajes fueron iniciados por miembros y usuarios ocasionales en cada una de las 24 horas del día.

Esto permite identificar los horarios de mayor actividad para cada segmento.

In [ ]:
# Crear tabla comparativa por hora

hour_summary = (
    rides_by_hour
    .pivot(
        index="start_hour",
        columns="member_casual",
        values="total_rides"
    )
    .reset_index()
)

hour_summary.columns.name = None

hour_summary["total_rides"] = (
    hour_summary["member"] +
    hour_summary["casual"]
)

hour_summary

,start_hour,casual,member,total_rides
0,0,44454,36089,80543
1,1,29168,21978,51146
2,2,18628,13070,31698
3,3,10576,8457,19033
4,4,8484,9547,18031
5,5,13300,37622,50922
6,6,28870,110283,139153
7,7,52776,218500,271276
8,8,76074,283416,359490
9,9,74154,178772,252926


In [ ]:
# Crear versión de presentación

hour_display = hour_summary.copy()

for col in ["member", "casual", "total_rides"]:
    hour_display[col] = (
        hour_display[col]
        .map("{:,}".format)
    )

hour_display

,start_hour,casual,member,total_rides
0,0,"44,454","36,089","80,543"
1,1,"29,168","21,978","51,146"
2,2,"18,628","13,070","31,698"
3,3,"10,576","8,457","19,033"
4,4,"8,484","9,547","18,031"
5,5,"13,300","37,622","50,922"
6,6,"28,870","110,283","139,153"
7,7,"52,776","218,500","271,276"
8,8,"76,074","283,416","359,490"
9,9,"74,154","178,772","252,926"


In [ ]:
# ---------------------------------------------------------
# HORAS DE MAYOR ACTIVIDAD
# ---------------------------------------------------------

member_peak_hours = (
    rides_by_hour[
        rides_by_hour["member_casual"] == "member"
    ]
    .sort_values(
        "total_rides",
        ascending=False
    )
    .head(5)
)

casual_peak_hours = (
    rides_by_hour[
        rides_by_hour["member_casual"] == "casual"
    ]
    .sort_values(
        "total_rides",
        ascending=False
    )
    .head(5)
)

print("Top 5 horas - Members")
display(member_peak_hours)

print("\nTop 5 horas - Casual")
display(casual_peak_hours)

Top 5 horas - Members


,start_hour,member_casual,total_rides
35,17,member,418434
33,16,member,366949
37,18,member,326529
17,8,member,283416
31,15,member,258670



Top 5 horas - Casual


,start_hour,member_casual,total_rides
34,17,casual,202674
32,16,casual,187609
36,18,casual,174312
30,15,casual,162810
28,14,casual,147152


**Identificación de horas pico**

Se ordenan los resultados de mayor a menor para identificar las cinco horas con mayor cantidad de viajes en cada segmento.

Esta comparación ayuda a determinar si miembros y usuarios ocasionales presentan los mismos horarios de mayor actividad o si sus patrones diarios son diferentes.

### Hallazgo 1 — Horas de mayor actividad

Ambos tipos de usuario alcanzan su mayor volumen de viajes durante la tarde.

Para los miembros (`member`), la hora de mayor actividad es **17:00**, con **418.434 viajes**, seguida por las **16:00** y las **18:00**.

Los usuarios ocasionales (`casual`) presentan un patrón similar durante la tarde: su máximo también ocurre a las **17:00**, con **202.674 viajes**, seguido por las **16:00**, **18:00**, **15:00** y **14:00**.

Sin embargo, aparece una diferencia importante entre ambos grupos: las **08:00** se encuentran entre las cinco horas de mayor actividad de los miembros, con **283.416 viajes**, mientras que ninguna hora de la mañana aparece entre las cinco principales de los usuarios ocasionales.

Esto indica que, aunque ambos segmentos presentan una fuerte actividad durante la tarde, los miembros muestran además un pico matutino claramente más pronunciado.

In [ ]:
# ---------------------------------------------------------
# PARTICIPACIÓN DE VIAJES ENTRE 06:00 Y 09:00
# ---------------------------------------------------------

morning_summary = (
    df.assign(
        morning_period=df["start_hour"].between(
            6,
            9,
            inclusive="both"
        )
    )
    .groupby("member_casual")
    .agg(
        total_rides=("ride_id", "count"),
        morning_rides=(
            "morning_period",
            "sum"
        )
    )
    .reset_index()
)

morning_summary["morning_percentage"] = (
    morning_summary["morning_rides"]
    / morning_summary["total_rides"]
    * 100
)

morning_summary["morning_percentage"] = (
    morning_summary["morning_percentage"]
    .round(2)
)

morning_summary

,member_casual,total_rides,morning_rides,morning_percentage
0,casual,2146638,231874,10.80
1,member,3885831,790971,20.36


In [ ]:
morning_display = morning_summary.copy()

morning_display["total_rides"] = (
    morning_display["total_rides"]
    .map("{:,}".format)
)

morning_display["morning_rides"] = (
    morning_display["morning_rides"]
    .map("{:,}".format)
)

morning_display["morning_percentage"] = (
    morning_display["morning_percentage"]
    .map("{:.2f}%".format)
)

morning_display

,member_casual,total_rides,morning_rides,morning_percentage
0,casual,"2,146,638","231,874",10.80%
1,member,"3,885,831","790,971",20.36%


**Participación de viajes durante la mañana**

Se calcula qué porcentaje de los viajes de cada tipo de usuario comienza entre las **06:00 y las 09:00**.

Esta comparación permite medir si alguno de los segmentos presenta una mayor concentración relativa de actividad durante las primeras horas del día.

### Hallazgo 2 — Uso durante las primeras horas del día

La diferencia entre ambos segmentos se vuelve más evidente al analizar los viajes iniciados entre las **06:00 y las 09:00**.

Los miembros realizaron **790.971 viajes** durante este período, equivalentes al **20,36 %** de todos sus viajes.

Los usuarios ocasionales realizaron **231.874 viajes** en el mismo horario, representando solamente el **10,80 %** de su actividad total.

Esto supone una diferencia de **9,56 puntos porcentuales** entre ambos grupos. En términos relativos, la proporción de viajes matutinos entre los miembros es casi el doble que entre los usuarios ocasionales.

Los resultados refuerzan la existencia de un patrón de uso diferente: los miembros presentan una mayor concentración de actividad durante las primeras horas de la mañana, mientras que los usuarios ocasionales concentran proporcionalmente más viajes desde el mediodía hacia la tarde.

Este comportamiento es compatible con un uso más rutinario del servicio entre los miembros, aunque el dataset no permite conocer directamente el motivo de cada viaje.

## 5. Duración de los viajes

En esta etapa se analiza la duración de los viajes realizados por miembros anuales (`member`) y usuarios ocasionales (`casual`).

El objetivo es determinar si ambos grupos utilizan las bicicletas durante períodos de tiempo similares o si existen diferencias relevantes en la duración de sus recorridos.

Para evitar que algunos viajes extremadamente largos distorsionen la interpretación, se comparan tanto la **media** como la **mediana**, además de distintos percentiles de la distribución.

In [ ]:
# ---------------------------------------------------------
# 5.1 DURACIÓN POR TIPO DE USUARIO
# ---------------------------------------------------------

duration_summary = (
    df.groupby("member_casual")
    ["ride_length_min"]
    .agg(
        total_rides="count",
        mean_duration="mean",
        median_duration="median",
        min_duration="min",
        max_duration="max"
    )
    .reset_index()
)

# Redondear resultados
duration_summary[
    [
        "mean_duration",
        "median_duration",
        "min_duration",
        "max_duration"
    ]
] = (
    duration_summary[
        [
            "mean_duration",
            "median_duration",
            "min_duration",
            "max_duration"
        ]
    ]
    .round(2)
)

duration_summary

,member_casual,total_rides,mean_duration,median_duration,min_duration,max_duration
0,casual,2146638,18.26,11.03,0.0,1439.98
1,member,3885831,12.01,8.57,0.0,1439.90


**Comparación de la duración de los viajes**

El código agrupa los viajes según el tipo de usuario y calcula distintas estadísticas descriptivas sobre `ride_length_min`.

La **media** muestra la duración promedio de los viajes, mientras que la **mediana** indica el valor que divide los viajes en dos mitades iguales.

La mediana es especialmente útil en este análisis porque resulta menos sensible a viajes excepcionalmente largos que podrían elevar artificialmente el promedio.

In [ ]:
# ---------------------------------------------------------
# 5.2 PERCENTILES DE DURACIÓN
# ---------------------------------------------------------

duration_percentiles = (
    df.groupby("member_casual")
    ["ride_length_min"]
    .quantile([0.25, 0.50, 0.75, 0.90])
    .unstack()
    .reset_index()
)

duration_percentiles.columns = [
    "member_casual",
    "p25",
    "p50_median",
    "p75",
    "p90"
]

for col in [
    "p25",
    "p50_median",
    "p75",
    "p90"
]:
    duration_percentiles[col] = (
        duration_percentiles[col]
        .round(2)
    )

duration_percentiles

,member_casual,p25,p50_median,p75,p90
0,casual,6.16,11.03,20.19,36.62
1,member,5.02,8.57,14.56,23.40


**Distribución de la duración mediante percentiles**

Los percentiles permiten observar cómo se distribuyen las duraciones sin depender únicamente del promedio.

- `p25`: el 25 % de los viajes dura hasta ese valor.
- `p50`: corresponde a la mediana.
- `p75`: el 75 % de los viajes dura hasta ese valor.
- `p90`: el 90 % de los viajes dura hasta ese valor.

Esta comparación ayuda a determinar si las diferencias entre miembros y usuarios ocasionales se mantienen a lo largo de toda la distribución de los viajes.

In [ ]:
# ---------------------------------------------------------
# DIFERENCIA ENTRE CASUAL Y MEMBER
# ---------------------------------------------------------

duration_compare = (
    duration_summary
    .set_index("member_casual")
)

mean_difference_pct = (
    (
        duration_compare.loc["casual", "mean_duration"]
        /
        duration_compare.loc["member", "mean_duration"]
    - 1)
    * 100
)

median_difference_pct = (
    (
        duration_compare.loc["casual", "median_duration"]
        /
        duration_compare.loc["member", "median_duration"]
    - 1)
    * 100
)

print(
    f"Los viajes casuales duran en promedio "
    f"{mean_difference_pct:.2f}% más que los viajes de miembros."
)

print(
    f"La mediana de duración de los casuales es "
    f"{median_difference_pct:.2f}% superior a la de los miembros."
)

Los viajes casuales duran en promedio 52.04% más que los viajes de miembros.
La mediana de duración de los casuales es 28.70% superior a la de los miembros.


### Hallazgo 1 — Los usuarios ocasionales realizan viajes más largos

La duración de los viajes presenta una diferencia clara entre ambos tipos de usuario.

Los usuarios ocasionales (`casual`) realizan viajes de **18,26 minutos en promedio**, frente a **12,01 minutos** entre los miembros (`member`). Esto significa que los viajes casuales duran aproximadamente **52,04 % más** en promedio.

La diferencia también se mantiene al utilizar la mediana, una medida menos sensible a valores extremos:

- **Casual:** 11,03 minutos.
- **Member:** 8,57 minutos.

La mediana de los usuarios ocasionales es **28,70 % superior**, lo que confirma que la diferencia no se debe únicamente a unos pocos viajes excepcionalmente largos.

Los percentiles refuerzan este patrón. Por ejemplo, el 90 % de los viajes de usuarios ocasionales dura hasta **36,62 minutos**, mientras que entre los miembros el mismo percentil se sitúa en **23,40 minutos**.

En conjunto, los resultados indican que los usuarios ocasionales tienden a utilizar las bicicletas durante períodos de tiempo más prolongados que los miembros anuales.

In [ ]:
# ---------------------------------------------------------
# 5.4 DURACIÓN POR USUARIO Y TIPO DE BICICLETA
# ---------------------------------------------------------

duration_by_bike = (
    df.groupby(
        [
            "member_casual",
            "rideable_type"
        ]
    )
    ["ride_length_min"]
    .agg(
        total_rides="count",
        mean_duration="mean",
        median_duration="median"
    )
    .reset_index()
)

duration_by_bike[
    [
        "mean_duration",
        "median_duration"
    ]
] = (
    duration_by_bike[
        [
            "mean_duration",
            "median_duration"
        ]
    ]
    .round(2)
)

duration_by_bike

,member_casual,rideable_type,total_rides,mean_duration,median_duration
0,casual,classic_bike,607938,29.00,15.98
1,casual,electric_bike,1538700,14.01,9.66
2,member,classic_bike,1277898,13.91,9.23
3,member,electric_bike,2607933,11.08,8.29


**Duración según el tipo de bicicleta**

Además de comparar la duración general, se analiza si las diferencias entre miembros y usuarios ocasionales cambian según el tipo de bicicleta utilizado.

Para cada combinación de usuario y bicicleta se calcula la cantidad de viajes, la duración promedio y la mediana.

Este análisis permite determinar si algún tipo de bicicleta está asociado con recorridos particularmente más largos en uno de los segmentos.

### Hallazgo 2 — La mayor diferencia aparece en las bicicletas clásicas

Al analizar la duración según el tipo de bicicleta, se observa que la diferencia entre usuarios ocasionales y miembros es especialmente pronunciada en las bicicletas clásicas (`classic_bike`).

Los usuarios ocasionales realizan viajes en bicicletas clásicas de **29,00 minutos en promedio**, mientras que los miembros registran un promedio de **13,91 minutos**.

La mediana presenta el mismo patrón:

- **Casual + classic bike:** 15,98 minutos.
- **Member + classic bike:** 9,23 minutos.

En las bicicletas eléctricas (`electric_bike`) la diferencia es considerablemente menor:

- **Casual:** 14,01 minutos de promedio.
- **Member:** 11,08 minutos de promedio.

Esto indica que los viajes prolongados de los usuarios ocasionales están especialmente asociados al uso de bicicletas clásicas, mientras que el comportamiento de ambos segmentos es más parecido cuando utilizan bicicletas eléctricas.

## 6. Preferencia de tipo de bicicleta

En esta etapa se analiza qué tipos de bicicleta utilizan con mayor frecuencia los miembros anuales (`member`) y los usuarios ocasionales (`casual`).

El objetivo es determinar si la elección del tipo de bicicleta constituye una diferencia relevante entre ambos segmentos o si sus preferencias son similares.

In [ ]:
# ---------------------------------------------------------
# 6.1 VIAJES POR TIPO DE BICICLETA
# ---------------------------------------------------------

rides_by_bike = (
    df.groupby(
        ["rideable_type", "member_casual"]
    )
    .size()
    .reset_index(name="total_rides")
)

rides_by_bike

,rideable_type,member_casual,total_rides
0,classic_bike,casual,607938
1,classic_bike,member,1277898
2,electric_bike,casual,1538700
3,electric_bike,member,2607933


**Distribución de viajes por tipo de bicicleta**

El código agrupa los viajes según el tipo de bicicleta (`rideable_type`) y el tipo de usuario (`member_casual`).

Posteriormente se cuenta cuántos viajes corresponden a cada combinación.

Esto permite comparar el volumen de uso de bicicletas clásicas y eléctricas entre miembros y usuarios ocasionales.

In [ ]:
# Crear Tabla Comparativa

bike_summary = (
    rides_by_bike
    .pivot(
        index="rideable_type",
        columns="member_casual",
        values="total_rides"
    )
    .reset_index()
)

bike_summary.columns.name = None

bike_summary["total_rides"] = (
    bike_summary["member"] +
    bike_summary["casual"]
)

bike_summary

,rideable_type,casual,member,total_rides
0,classic_bike,607938,1277898,1885836
1,electric_bike,1538700,2607933,4146633


In [ ]:
# Crear versión de presentación

bike_display = bike_summary.copy()

for col in ["member", "casual", "total_rides"]:
    bike_display[col] = (
        bike_display[col]
        .map("{:,}".format)
    )

bike_display

,rideable_type,casual,member,total_rides
0,classic_bike,"607,938","1,277,898","1,885,836"
1,electric_bike,"1,538,700","2,607,933","4,146,633"


In [ ]:
# ---------------------------------------------------------
# 6.3 PARTICIPACIÓN PORCENTUAL POR TIPO DE USUARIO
# ---------------------------------------------------------

bike_percentage = rides_by_bike.copy()

bike_percentage["percentage"] = (
    bike_percentage["total_rides"]
    / bike_percentage.groupby(
        "member_casual"
    )["total_rides"].transform("sum")
    * 100
)

bike_percentage["percentage"] = (
    bike_percentage["percentage"]
    .round(2)
)

bike_percentage

,rideable_type,member_casual,total_rides,percentage
0,classic_bike,casual,607938,28.32
1,classic_bike,member,1277898,32.89
2,electric_bike,casual,1538700,71.68
3,electric_bike,member,2607933,67.11


**Participación de cada tipo de bicicleta dentro de cada segmento**

Para comparar correctamente las preferencias de ambos grupos, se calcula qué porcentaje de los viajes de cada tipo de usuario corresponde a cada categoría de bicicleta.

Esto permite evaluar la preferencia relativa sin que la diferencia en el tamaño total de los segmentos distorsione la comparación.

In [ ]:
# Tabla Final de porcentajes

bike_percentage_display = (
    bike_percentage
    .pivot(
        index="rideable_type",
        columns="member_casual",
        values="percentage"
    )
    .reset_index()
)

bike_percentage_display.columns.name = None

bike_percentage_display["member"] = (
    bike_percentage_display["member"]
    .map("{:.2f}%".format)
)

bike_percentage_display["casual"] = (
    bike_percentage_display["casual"]
    .map("{:.2f}%".format)
)

bike_percentage_display

,rideable_type,casual,member
0,classic_bike,28.32%,32.89%
1,electric_bike,71.68%,67.11%


### Hallazgo  — Preferencia por bicicletas eléctricas

Tanto los miembros anuales (`member`) como los usuarios ocasionales (`casual`) muestran una clara preferencia por las bicicletas eléctricas.

Entre los usuarios ocasionales, el **71,68 %** de los viajes se realiza en `electric_bike`, mientras que las bicicletas clásicas representan el **28,32 %**.

En el caso de los miembros, el **67,11 %** de los viajes corresponde a bicicletas eléctricas y el **32,89 %** a bicicletas clásicas.

Aunque los usuarios ocasionales presentan una preferencia ligeramente mayor por las bicicletas eléctricas, la diferencia entre ambos segmentos es relativamente pequeña.

Esto indica que el tipo de bicicleta no constituye uno de los principales factores que diferencian el comportamiento de miembros y usuarios ocasionales. Las diferencias más relevantes aparecen en otras dimensiones, como la temporalidad, la duración de los viajes y el uso durante fines de semana.

**Conclusión:** La preferencia de vehículo es similar en ambos grupos, por lo que esta variable tiene menor poder explicativo que los patrones temporales y de duración observados anteriormente.

## 7. Estaciones y comportamiento geográfico

En esta etapa se analiza la distribución de los viajes según la estación de inicio.

El objetivo es identificar qué estaciones concentran un mayor volumen de viajes y, especialmente, cuáles presentan una mayor proporción de usuarios ocasionales (`casual`).

Este análisis puede ayudar a detectar ubicaciones estratégicas para futuras acciones de marketing orientadas a la conversión de usuarios ocasionales en miembros anuales.

In [ ]:
# ---------------------------------------------------------
# 7.1 DATASET PARA ANÁLISIS DE ESTACIONES
# ---------------------------------------------------------

df_stations = df.dropna(
    subset=["start_station_name"]
).copy()

print(
    f"Viajes totales del dataset: {len(df):,}"
)

print(
    f"Viajes con estación de inicio identificada: "
    f"{len(df_stations):,}"
)

station_coverage = (
    len(df_stations) / len(df) * 100
)

print(
    f"Cobertura del análisis geográfico: "
    f"{station_coverage:.2f}%"
)

Viajes totales del dataset: 6,032,469
Viajes con estación de inicio identificada: 4,759,310
Cobertura del análisis geográfico: 78.89%


**Preparación del análisis geográfico**

No todos los registros contienen información sobre la estación de inicio.

Por este motivo se crea un DataFrame independiente, `df_stations`, que conserva únicamente los viajes donde `start_station_name` está disponible.

Los registros sin estación identificada se mantienen en el dataset principal, ya que siguen siendo válidos para otros análisis como duración, horario, día de la semana o tipo de bicicleta.

In [ ]:
# ---------------------------------------------------------
# 7.2 ESTACIONES CON MAYOR VOLUMEN TOTAL
# ---------------------------------------------------------

top_stations_total = (
    df_stations
    .groupby("start_station_name")
    .size()
    .reset_index(name="total_rides")
    .sort_values(
        "total_rides",
        ascending=False
    )
    .head(10)
)

top_stations_total

,start_station_name,total_rides
848,Navy Pier,71171
339,DuSable Lake Shore Dr & Monroe St,41168
794,Michigan Ave & Oak St,36529
340,DuSable Lake Shore Dr & North Blvd,35106
153,Canal St & Madison St,29602
1792,State St & Chicago Ave,29309
1891,Wells St & Concord Ln,29256
1820,Theater on the Lake,28437
1892,Wells St & Elm St,27806
802,Millennium Park,26826


**Estaciones con mayor número de viajes**

Se agrupan los viajes según la estación de inicio y se cuentan los registros correspondientes a cada ubicación.

Posteriormente se seleccionan las diez estaciones con mayor volumen total de viajes.

Esto permite identificar cuáles son los puntos de mayor actividad dentro del sistema.

In [ ]:
# ---------------------------------------------------------
# 7.3 TOP ESTACIONES PARA USUARIOS CASUALES
# ---------------------------------------------------------

top_casual_stations = (
    df_stations[
        df_stations["member_casual"] == "casual"
    ]
    .groupby("start_station_name")
    .size()
    .reset_index(name="casual_rides")
    .sort_values(
        "casual_rides",
        ascending=False
    )
    .head(10)
)

top_casual_stations

,start_station_name,casual_rides
847,Navy Pier,54760
338,DuSable Lake Shore Dr & Monroe St,32478
793,Michigan Ave & Oak St,22598
339,DuSable Lake Shore Dr & North Blvd,19711
801,Millennium Park,18192
1598,Shedd Aquarium,17071
1711,Theater on the Lake,16192
342,Dusable Harbor,14692
787,Michigan Ave & 8th St,11006
820,Montrose Harbor,10515


**Estaciones con mayor volumen de usuarios ocasionales**

Se filtran únicamente los viajes realizados por usuarios `casual` y se cuentan los viajes iniciados en cada estación.

Este análisis permite detectar qué estaciones concentran el mayor número absoluto de viajes ocasionales.

In [ ]:
# ---------------------------------------------------------
# 7.4 PROPORCIÓN DE USUARIOS CASUALES POR ESTACIÓN
# ---------------------------------------------------------

station_user_summary = (
    df_stations
    .groupby(
        ["start_station_name", "member_casual"]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

station_user_summary["total_rides"] = (
    station_user_summary["member"] +
    station_user_summary["casual"]
)

station_user_summary["casual_percentage"] = (
    station_user_summary["casual"]
    / station_user_summary["total_rides"]
    * 100
)

station_user_summary["casual_percentage"] = (
    station_user_summary["casual_percentage"]
    .round(2)
)

station_user_summary.head()

member_casual,start_station_name,casual,member,total_rides,casual_percentage
0,11th St Ped Bridge,1,0,1,100.00
1,2112 W Peterson Ave,244,579,823,29.65
2,21st St & Pulaski Rd,58,32,90,64.44
3,63rd St Beach,881,434,1315,67.00
4,900 W Harrison St,2717,5879,8596,31.61


**Participación de usuarios ocasionales por estación**

Para cada estación se calcula el número de viajes realizados por miembros y usuarios ocasionales.

A partir de estos valores se obtiene el porcentaje de viajes `casual` sobre el total de viajes iniciados en cada estación.

Esta métrica es más informativa que el volumen absoluto, ya que permite identificar estaciones donde los usuarios ocasionales representan una proporción especialmente elevada de la actividad.

In [ ]:
# ---------------------------------------------------------
# 7.5 ESTACIONES CON ALTA PROPORCIÓN CASUAL
#     Y VOLUMEN SIGNIFICATIVO
# ---------------------------------------------------------

MIN_RIDES = 5000

high_casual_stations = (
    station_user_summary[
        station_user_summary["total_rides"] >= MIN_RIDES
    ]
    .sort_values(
        "casual_percentage",
        ascending=False
    )
    .head(15)
)

high_casual_stations

member_casual,start_station_name,casual,member,total_rides,casual_percentage
1707,Shedd Aquarium,17071,3793,20864,81.82
389,Field Museum,10325,2413,12738,81.06
339,DuSable Lake Shore Dr & Monroe St,32478,8690,41168,78.89
848,Navy Pier,54760,16411,71171,76.94
19,Adler Planetarium,10483,3989,14472,72.44
770,McCormick Place,5032,1927,6959,72.31
343,Dusable Harbor,14692,5691,20383,72.08
802,Millennium Park,18192,8634,26826,67.81
788,Michigan Ave & 8th St,11006,6397,17403,63.24
794,Michigan Ave & Oak St,22598,13931,36529,61.86


**Filtro por volumen mínimo**

Para evitar interpretar como relevantes estaciones con muy pocos viajes, se establece un mínimo de **5.000 viajes** por estación.

De esta forma, las estaciones seleccionadas presentan tanto una elevada proporción de usuarios ocasionales como un volumen suficiente para que el patrón resulte significativo desde una perspectiva de negocio.

In [ ]:
# ---------------------------------------------------------
# 7.6 TOP ESTACIONES PARA MIEMBROS
# ---------------------------------------------------------

top_member_stations = (
    df_stations[
        df_stations["member_casual"] == "member"
    ]
    .groupby("start_station_name")
    .size()
    .reset_index(name="member_rides")
    .sort_values(
        "member_rides",
        ascending=False
    )
    .head(10)
)

top_member_stations

,start_station_name,member_rides
151,Canal St & Madison St,22971
1689,State St & Chicago Ave,21637
1788,Wells St & Concord Ln,19675
1789,Wells St & Elm St,19651
234,Clinton St & Jackson Blvd,19622
236,Clinton St & Madison St,19528
1792,Wells St & Huron St,18132
210,Clark St & Elm St,17587
240,Clinton St & Washington Blvd,17557
1791,Wells St & Hubbard St,16868


### Hallazgo 1 — Concentración geográfica de usuarios ocasionales

El análisis de estaciones muestra que los usuarios ocasionales no se distribuyen de manera uniforme dentro del sistema.

La estación con mayor cantidad de viajes `casual` es **Navy Pier**, con **54.760 viajes**, seguida por **DuSable Lake Shore Dr & Monroe St** con **32.478** y **Michigan Ave & Oak St** con **22.598**.

Sin embargo, el volumen absoluto no es la única diferencia relevante. Al analizar la proporción de usuarios ocasionales dentro de cada estación, aparecen concentraciones especialmente elevadas:

- **Shedd Aquarium:** 81,82 % casual.
- **Field Museum:** 81,06 % casual.
- **DuSable Lake Shore Dr & Monroe St:** 78,89 % casual.
- **Navy Pier:** 76,94 % casual.
- **Adler Planetarium:** 72,44 % casual.
- **DuSable Harbor:** 72,08 % casual.

Estos resultados indican que determinadas estaciones concentran tanto un volumen significativo como una proporción muy elevada de usuarios ocasionales.

Desde una perspectiva de negocio, estas ubicaciones representan puntos especialmente interesantes para orientar campañas de conversión hacia membresías anuales.

### Hallazgo 2 — Miembros y usuarios ocasionales se concentran en estaciones diferentes

Las estaciones más utilizadas por los miembros presentan un patrón distinto al observado entre los usuarios ocasionales.

Entre los miembros, las estaciones con mayor número de viajes incluyen:

- **Canal St & Madison St:** 22.971 viajes.
- **State St & Chicago Ave:** 21.637.
- **Wells St & Concord Ln:** 19.675.
- **Wells St & Elm St:** 19.651.
- **Clinton St & Jackson Blvd:** 19.622.

En cambio, entre los usuarios ocasionales destacan estaciones como Navy Pier, Millennium Park, Shedd Aquarium y diferentes puntos de DuSable Lake Shore Drive.

Esto demuestra que la ubicación constituye otra dimensión relevante en las diferencias de comportamiento entre ambos segmentos.

Varias de las estaciones con alta presencia `casual` están asociadas, por su nombre, a parques, museos, zonas costeras o lugares de interés, lo que es compatible con el patrón de mayor actividad recreativa observado previamente durante fines de semana y períodos de mayor demanda.

No obstante, el dataset no contiene el propósito declarado del viaje, por lo que esta relación debe considerarse una interpretación basada en los patrones observados.

**Limitación del análisis geográfico**

De los **6.032.469 viajes** del dataset, **4.759.310** contienen una estación de inicio identificada.

Por lo tanto, el análisis geográfico cubre el **78,89 %** de los viajes, mientras que aproximadamente el **21,11 %** restante no dispone de `start_station_name`.

Los viajes sin estación identificada se conservaron en el dataset principal porque siguen siendo válidos para los demás análisis.

# 📤 Etapa 3 — Preparación y exportación de datos para Tableau

Una vez finalizado el análisis exploratorio, se preparan datasets agregados específicamente para la visualización en Tableau.

El objetivo es reducir considerablemente el volumen de datos sin perder las métricas necesarias para representar los principales hallazgos del análisis.

Se generan cuatro archivos:

1. **Temporalidad:** mes, día de la semana y hora.
2. **Duración:** media, mediana y percentiles.
3. **Tipo de bicicleta:** volumen y participación porcentual.
4. **Estaciones:** ubicación, volumen y composición de usuarios.

In [ ]:
tableau_path = os.path.join(
    OUTPUT_DIR,
    "Cyclistic_Tableau"
)

os.makedirs(
    tableau_path,
    exist_ok=True
)

print(
    f"Carpeta preparada:\n"
    f"{tableau_path}"
)


In [ ]:
# ---------------------------------------------------------
# TABLEAU 1 — TEMPORALIDAD
# ---------------------------------------------------------

tableau_time = (
    df.groupby(
        [
            "month_start",
            "year_month",
            "day_of_week",
            "day_num",
            "start_hour",
            "is_weekend",
            "member_casual"
        ],
        observed=True
    )
    .size()
    .reset_index(name="total_rides")
)

# Crear etiqueta más clara para Tableau
tableau_time["week_part"] = (
    tableau_time["is_weekend"]
    .map({
        False: "Weekday",
        True: "Weekend"
    })
)

# Ordenar cronológicamente
tableau_time = tableau_time.sort_values(
    [
        "month_start",
        "day_num",
        "start_hour",
        "member_casual"
    ]
)

time_file = os.path.join(
    tableau_path,
    "cyclistic_tableau_time.csv"
)

tableau_time.to_csv(
    time_file,
    index=False
)

print(
    f"Temporalidad: "
    f"{len(tableau_time):,} filas"
)

Temporalidad: 4,032 filas


### Dataset de temporalidad

Los viajes se agregan según mes, día de la semana, hora y tipo de usuario.

Este nivel de agregación permite construir diferentes visualizaciones temporales en Tableau sin necesidad de trabajar con millones de registros individuales.

Se conserva `month_start` como una fecha real y `day_num` como variable de orden para garantizar que meses y días se representen cronológicamente.

In [ ]:
# ---------------------------------------------------------
# TABLEAU 2 — DURACIÓN
# ---------------------------------------------------------

def duration_stats(data, group_cols):

    result = (
        data.groupby(group_cols)["ride_length_min"]
        .agg(
            total_rides="count",
            mean_duration="mean",
            median_duration="median"
        )
        .reset_index()
    )

    percentiles = (
        data.groupby(group_cols)["ride_length_min"]
        .quantile([0.25, 0.75, 0.90])
        .unstack()
        .reset_index()
    )

    percentiles = percentiles.rename(
        columns={
            0.25: "p25",
            0.75: "p75",
            0.90: "p90"
        }
    )

    return result.merge(
        percentiles,
        on=group_cols
    )


# Duración general por usuario
duration_user = duration_stats(
    df,
    ["member_casual"]
)

duration_user["rideable_type"] = "All"
duration_user["analysis_level"] = "User overall"


# Duración por usuario y bicicleta
duration_bike = duration_stats(
    df,
    [
        "member_casual",
        "rideable_type"
    ]
)

duration_bike["analysis_level"] = "User by bike"


# Unir ambos niveles
tableau_duration = pd.concat(
    [
        duration_user,
        duration_bike
    ],
    ignore_index=True
)


# Redondear únicamente valores numéricos
duration_cols = [
    "mean_duration",
    "median_duration",
    "p25",
    "p75",
    "p90"
]

tableau_duration[duration_cols] = (
    tableau_duration[duration_cols]
    .round(2)
)

duration_file = os.path.join(
    tableau_path,
    "cyclistic_tableau_duration.csv"
)

tableau_duration.to_csv(
    duration_file,
    index=False
)

tableau_duration

,member_casual,total_rides,mean_duration,median_duration,p25,p75,p90,rideable_type,analysis_level
0,casual,2146638,18.26,11.03,6.16,20.19,36.62,All,User overall
1,member,3885831,12.01,8.57,5.02,14.56,23.40,All,User overall
2,casual,607938,29.00,15.98,8.86,30.32,58.04,classic_bike,User by bike
3,casual,1538700,14.01,9.66,5.47,17.00,28.84,electric_bike,User by bike
4,member,1277898,13.91,9.23,5.35,16.48,27.47,classic_bike,User by bike
5,member,2607933,11.08,8.29,4.86,13.78,21.48,electric_bike,User by bike


In [ ]:
# ---------------------------------------------------------
# TABLEAU 3 — TIPO DE BICICLETA
# ---------------------------------------------------------

tableau_bikes = (
    df.groupby(
        [
            "member_casual",
            "rideable_type"
        ]
    )
    .size()
    .reset_index(name="total_rides")
)

tableau_bikes["percentage_within_user"] = (
    tableau_bikes["total_rides"]
    / tableau_bikes.groupby(
        "member_casual"
    )["total_rides"].transform("sum")
    * 100
)

tableau_bikes["percentage_within_user"] = (
    tableau_bikes["percentage_within_user"]
    .round(2)
)

bikes_file = os.path.join(
    tableau_path,
    "cyclistic_tableau_bikes.csv"
)

tableau_bikes.to_csv(
    bikes_file,
    index=False
)

tableau_bikes

,member_casual,rideable_type,total_rides,percentage_within_user
0,casual,classic_bike,607938,28.32
1,casual,electric_bike,1538700,71.68
2,member,classic_bike,1277898,32.89
3,member,electric_bike,2607933,67.11


In [ ]:
# ---------------------------------------------------------
# TABLEAU 4 — ESTACIONES Y MAPA
# ---------------------------------------------------------

df_stations = df.dropna(
    subset=["start_station_name"]
).copy()


# Conteo por estación y usuario
station_counts = (
    df_stations
    .groupby(
        [
            "start_station_name",
            "member_casual"
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)


station_counts["total_rides"] = (
    station_counts["member"]
    +
    station_counts["casual"]
)


station_counts["casual_percentage"] = (
    station_counts["casual"]
    / station_counts["total_rides"]
    * 100
)


station_counts["member_percentage"] = (
    station_counts["member"]
    / station_counts["total_rides"]
    * 100
)


# Coordenadas promedio de cada estación
station_coordinates = (
    df_stations
    .dropna(
        subset=[
            "start_lat",
            "start_lng"
        ]
    )
    .groupby("start_station_name")
    .agg(
        start_lat=("start_lat", "mean"),
        start_lng=("start_lng", "mean")
    )
    .reset_index()
)


# Combinar métricas + coordenadas
tableau_stations = station_counts.merge(
    station_coordinates,
    on="start_station_name",
    how="left"
)


tableau_stations[
    [
        "casual_percentage",
        "member_percentage"
    ]
] = (
    tableau_stations[
        [
            "casual_percentage",
            "member_percentage"
        ]
    ]
    .round(2)
)


stations_file = os.path.join(
    tableau_path,
    "cyclistic_tableau_stations.csv"
)

tableau_stations.to_csv(
    stations_file,
    index=False
)

print(
    f"Estaciones exportadas: "
    f"{len(tableau_stations):,}"
)

Estaciones exportadas: 1,979


In [ ]:
# ---------------------------------------------------------
# VALIDACIÓN DE ARCHIVOS TABLEAU
# ---------------------------------------------------------

print("==============================")
print("VALIDACIÓN TABLEAU")
print("==============================")

print(
    f"Dataset original limpio: "
    f"{len(df):,}"
)

print(
    f"Viajes en tableau_time: "
    f"{tableau_time['total_rides'].sum():,}"
)

print(
    f"Viajes en tableau_bikes: "
    f"{tableau_bikes['total_rides'].sum():,}"
)

print(
    f"Viajes con estación identificada: "
    f"{tableau_stations['total_rides'].sum():,}"
)

print(
    f"\nFilas tableau_time: "
    f"{len(tableau_time):,}"
)

print(
    f"Filas tableau_duration: "
    f"{len(tableau_duration):,}"
)

print(
    f"Filas tableau_bikes: "
    f"{len(tableau_bikes):,}"
)

print(
    f"Filas tableau_stations: "
    f"{len(tableau_stations):,}"
)

VALIDACIÓN TABLEAU
Dataset original limpio: 6,032,469
Viajes en tableau_time: 6,032,469
Viajes en tableau_bikes: 6,032,469
Viajes con estación identificada: 4,759,310

Filas tableau_time: 4,032
Filas tableau_duration: 6
Filas tableau_bikes: 4
Filas tableau_stations: 1,979


In [ ]:
# Controles automáticos

assert (
    tableau_time["total_rides"].sum()
    == len(df)
)

assert (
    tableau_bikes["total_rides"].sum()
    == len(df)
)

print(
    "\n✅ Los datasets temporales y de bicicleta "
    "reproducen correctamente el total de viajes."
)


✅ Los datasets temporales y de bicicleta reproducen correctamente el total de viajes.


In [ ]:
print("\nArchivos preparados para Tableau:\n")

for file in os.listdir(tableau_path):

    full_path = os.path.join(
        tableau_path,
        file
    )

    size_kb = (
        os.path.getsize(full_path)
        / 1024
    )

    print(
        f"{file} → {size_kb:.2f} KB"
    )


Archivos preparados para Tableau:

cyclistic_tableau_time.csv → 223.77 KB
cyclistic_tableau_duration.csv → 0.48 KB
cyclistic_tableau_bikes.csv → 0.20 KB
cyclistic_tableau_stations.csv → 162.12 KB
